# Demo — Mapping a Federated Platform to a Governance Operating Model

**Trailhead Provisions** grew by acquisition into four domain teams sharing a data platform.
Before writing a single rule or access policy, a data architect must establish the **operating
model**: which decisions stay central, which belong to domain teams, and who is **Accountable**
for each governance function.

This walkthrough inspects the platform's domains and cross-domain dependencies, models one row
of a RACI matrix, and sketches a slice of the governance topology; the exercise asks you to
justify the full governance model, complete the RACI across all functions, and draw the complete
topology.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import GovernedCatalog

gc = GovernedCatalog("trailhead.db")
inventory = pd.DataFrame([
    {
        "domain": gc.catalog.table(t).domain,
        "table": t,
        "owner": gc.catalog.table(t).owner or "(no owner set)",
        "classification": gc.catalog.table(t).classification or "(unset)",
    }
    for t in gc.tables()
]).sort_values(["domain", "table"]).reset_index(drop=True)
inventory

## Why pure decentralization fails under GDPR & CSRD
Each domain team runs its own pipelines, yet regulations cross domain boundaries. Inspect the
platform's cross-domain semantic conflicts and pipeline jobs to see where domains collide:

In [ ]:
print("--- Cross-domain semantic conflicts ---")
display(gc.semantic_conflicts()[["concept", "domain_a", "domain_b", "impact"]])

print("\n--- Cross-domain pipeline jobs ---")
gc.lineage.events()

## How to structure a RACI row (the single-Accountable rule)
In a **federated / hybrid** model, domains are **Responsible (`R`)** for executing governance
on their own tables, but every governance function must have **exactly one Accountable (`A`)**
owner so accountability never falls between chairs.

Here is how one governance function row looks before you build the full matrix in the exercise:

| Governance function | E-comm Orders | Customer Identity | Inventory & Fulfillment | Marketing & Campaigns | Central Governance Lead |
|---|---|---|---|---|---|
| **Classification taxonomy (`PII`/`SENSITIVE`/`PUBLIC`)** | `R` (tag order tables) | `R` (tag customer PII) | `R` (tag cost/carbon) | `R` (tag campaign tables) | **`A`** (owns standard & audits) |

## How to sketch the topology in Mermaid
Use a Mermaid `graph TD` block to show how central standards flow down to domains while regulated
data flows across domains. Here is a two-domain slice illustrating the pattern:

```mermaid
graph TD
    Core["Central Governance Core<br/>(Taxonomy • Contract Schema • Audit)"]
    Cust["Customer Identity & Loyalty<br/>(customer)"]
    Mkt["Marketing & Campaigns<br/>(campaign, campaign_membership)"]

    Core -->|"guardrails & PII standards"| Cust
    Core -->|"guardrails & PII standards"| Mkt
    Cust -->|"customer_id + marketing_consent (SoR)"| Mkt
```

**Takeaway / your turn:** in the exercise you'll state and justify Trailhead's governance model,
build the complete RACI across all governance functions (with one `A` per row), and draw the
full four-domain topology including both the GDPR consent flow and the CSRD carbon reporting flow.